In [24]:
import numpy as np
import pandas as pd
from sympy import limit
import wandb

api = wandb.Api(
    api_key="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",
    timeout=60
)

# 1. Get last 10 runs (sorted by creation time descending)
runs = api.runs(
    "eibl-usc/graph-clip",
    filters={
        "display_name": {"$regex": "train3.+"},
        # "config.dataset": "ukr_rus_twitter",
        # "config.task_name": "neighbor_matching",
    },
    order="-created_at",
    per_page=30,
    # limit to 10:
    lazy=False
)

In [28]:
import itertools
runs_ = list(itertools.islice(runs, 30))

rows = []
for run in runs_:
    attrs = getattr(run, "_attrs", {}) or {}
    params = ((attrs.get("config") or {}).get("params") or {})
    summary = attrs.get("summaryMetrics") or {}

    rows.append({
        "run_id": attrs.get("name"),
        "display_name": attrs.get("displayName"),
        "state": attrs.get("state"),
        "dataset": params.get("dataset"),
        "task_name": params.get("task_name"),
        "prefix": params.get("prefix"),
        "pretrained_model_run": params.get("pretrained_model_run"),
        "n_shots": params.get("n_shots"),
        "n_way": params.get("n_way"),
        "n_query": params.get("n_query"),
        "zero_shot": params.get("zero_shot"),
        "test_accuracy": summary.get("test_accuracy"),
        "train_accuracy": summary.get("train_accuracy"),
        "test_f1": summary.get("test_f1"),
        "test_roc_auc": summary.get("test_roc_auc"),
        "created_at": attrs.get("createdAt"),
        'steps': attrs['historyKeys']['keys'].get('_step', {}).get('typeCounts', [{}])[0].get('count', np.nan),
        'tags': attrs.get("tags", []),
    })
df = pd.DataFrame(rows)
df["train1_dataset"] = df["pretrained_model_run"].str.extract(r"train1_(ukr_rus_twitter|midterm|covid19_twitter)_")
df["train1_task"] = df["pretrained_model_run"].str.extract(r"train1_.+?_(nm|pl|lp)_")
df["eval1_task"] = df["task_name"].map({
    "neighbor_matching": "nm",
    "temporal_link_prediction": "lp",
    "classification": "pl",
})
df['created_at'] = pd.to_datetime(df['created_at'])
df["shot_label"] = df.apply(lambda r: 0 if bool(r.get("zero_shot", False)) else r["n_shots"], axis=1)
# df['is_eval'] = df['display_name'].str.contains(r"eval")
# plot_df = df[df["eval1_task"].isin(EVAL_TASKS) & df["train1_task"].eq("nm")].copy()
df['eval1_dataset'] = df['dataset']
df['trained_on_display_name'] = df.pretrained_model_run.str.split('/').str[1]
df['month/day'] = df['created_at'].dt.month.astype(str) + '/' + df['created_at'].dt.day.astype(str)
df = df.sort_values('created_at', ascending=False)
d = {
    'trained_on_n_way': 'n_way',
    'trained_on_n_query': 'n_query',
    'trained_on_n_shots': 'n_shots',
    'trained_on_steps': 'steps',
}
mask = df['trained_on_display_name'].isin(df['display_name'])
existing_trained_on_display_names = df.trained_on_display_name[mask]
dname2stats = df.set_index('display_name').loc[existing_trained_on_display_names][list(d.values())]
for col, stat in d.items():
    df[col] = df.trained_on_display_name.map(dname2stats[stat].to_dict())
df__ = df.copy()

df = df[df.state.ne('running')]
df['auc'] = df.test_roc_auc

In [46]:
df.to_csv('train3_apr_29.csv', index=False)

In [9]:
df = pd.read_csv('train3_apr_29.csv')

In [ ]:
df[df.test_roc_auc.notna()][['prefix', 'dataset', 'task_name', 'n_shots', 'auc']].to_csv('tr')

'\tprefix\tdataset\ttask_name\tn_shots\tauc\n0\ttrain3_exp3_covid_nm_to_ukr_rus_nm_to_election2020_nm\telection2020\tneighbor_matching\t3\t0.884956940185185\n2\ttrain3_exp1_midterm_nm_to_covid_nm_to_ukr_rus_nm\tukr_rus_twitter\tneighbor_matching\t3\t0.99230570625\n4\ttrain3_exp3_covid_nm_to_ukr_rus_nm_to_covid_political_nm\tcovid_political\tneighbor_matching\t3\t0.9165226269444444\n6\ttrain3_exp2_midterm_nm_to_ukr_rus_nm_to_covid_nm\tcovid19_twitter\tneighbor_matching\t3\t0.9978122875\n10\ttrain3_exp3_covid_nm_to_ukr_rus_nm_to_ukr_rus_suspended_nm\tukr_rus_suspended\tneighbor_matching\t3\t0.9241267305555556\n8\ttrain3_exp1_midterm_nm_to_covid_nm_to_election2020_nm\telection2020\tneighbor_matching\t3\t0.8862653059259259\n9\ttrain3_exp3_covid_nm_to_ukr_rus_nm_to_midterm_nm\tmidterm\tneighbor_matching\t3\t0.9676111052083334\n17\ttrain3_exp2_midterm_nm_to_ukr_rus_nm_to_ukr_rus_suspended_nm\tukr_rus_suspended\tneighbor_matching\t3\t0.9364147680555556\n12\ttrain3_exp1_midterm_nm_to_covid_nm_